## 1. Imports

In [11]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.callbacks import TensorBoard

## 2. Configuration and hyperparameters

In [ ]:
DATASET_ROOT = Path("CV9/bird_dataset")
TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "validation"
TEST_DIR = DATASET_ROOT / "test"

IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS = 64, 64, 3
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 1e-3
PATIENCE = 5
DROPOUT_RATE = 0.3
L1_FACTOR = 1e-6
L2_FACTOR = 1e-4
RANDOM_SEED = 42

tf.keras.utils.set_random_seed(RANDOM_SEED)

for name, path in {"train": TRAIN_DIR, "validation": VAL_DIR, "test": TEST_DIR}.items():
    if not path.is_dir():
        raise FileNotFoundError(
            f"{name} directory not found: {path.resolve()}. "
            "Update DATASET_ROOT to your dataset location."
        )

print("TensorFlow:", tf.__version__)
print("Dataset:", DATASET_ROOT.resolve())

## 3. Create image generators

Training images are rescaled and randomly augmented on the fly. Validation and test images are only rescaled.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    str(TRAIN_DIR), target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    color_mode="rgb", batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True, seed=RANDOM_SEED,
)
val_gen = eval_datagen.flow_from_directory(
    str(VAL_DIR), target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    color_mode="rgb", batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)
test_gen = eval_datagen.flow_from_directory(
    str(TEST_DIR), target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    color_mode="rgb", batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)

class_names = list(train_gen.class_indices.keys())
NUM_CLASSES = len(class_names)

if train_gen.class_indices != val_gen.class_indices:
    raise ValueError("Train and validation class mappings differ.")
if train_gen.class_indices != test_gen.class_indices:
    raise ValueError("Train and test class mappings differ.")

print("Classes:", NUM_CLASSES)
print("Training images:", train_gen.samples)
print("Validation images:", val_gen.samples)
print("Test images:", test_gen.samples)
print("Batches per training epoch:", len(train_gen))

## 4. Inspect one augmented batch

Run this cell again to see a different random set of augmentations.

In [ ]:
images, labels = next(train_gen)
print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Pixel range:", float(images.min()), "to", float(images.max()))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(np.clip(images[i], 0, 1))
        ax.set_title(class_names[int(np.argmax(labels[i]))])
    ax.axis("off")
plt.tight_layout()
plt.show()
train_gen.reset()

## 5. Build the CNN

Each convolutional block uses Conv2D → BatchNormalization → ReLU → MaxPooling. L1/L2 penalties are applied to convolutional kernels and the hidden Dense layer.

In [ ]:
def build_cnn(num_classes):
    kernel_reg = regularizers.l1_l2(l1=L1_FACTOR, l2=L2_FACTOR)

    model = models.Sequential([
        layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS)),

        layers.Conv2D(32, (3, 3), padding="valid",
                      use_bias=False, kernel_regularizer=kernel_reg),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding="valid",
                      use_bias=False, kernel_regularizer=kernel_reg),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((3, 3)),

        layers.Dropout(DROPOUT_RATE),

        layers.Conv2D(64, (3, 3), padding="valid",
                      use_bias=False, kernel_regularizer=kernel_reg),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((3, 3)),

        layers.Flatten(),
        layers.Dense(64, activation="relu", kernel_regularizer=kernel_reg),
        layers.Dropout(DROPOUT_RATE),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_classifier")

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_cnn(NUM_CLASSES)
model.summary()

## 6. Train the model

`model.fit()` obtains batches from `train_gen` automatically. Keras infers the number of batches from the generator.

In [ ]:
checkpoint_path = Path("best_cnn_model.h5")

early_stopping = EarlyStopping(
    monitor="val_loss", patience=PATIENCE,
    restore_best_weights=True, mode="min",
)
checkpoint = ModelCheckpoint(
    filepath=str(checkpoint_path), monitor="val_loss",
    save_best_only=True, mode="min", verbose=1,
)


tensorboard = TensorBoard(log_dir="./logs", update_freq="epoch")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[early_stopping, checkpoint, tensorboard],
    verbose=1,
)

## 7. Plot training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set(title="Loss", xlabel="Epoch", ylabel="Loss")
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Evaluate on the test set

The test set is used only for final evaluation, after model selection using validation loss.

In [ ]:
if checkpoint_path.exists():
    best_model = tf.keras.models.load_model(checkpoint_path)
else:
    best_model = model

test_loss, test_accuracy = best_model.evaluate(test_gen, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 9. Confusion matrix (optional)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

test_gen.reset()
probabilities = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
y_true = test_gen.classes

cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(12, 12))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
display.plot(ax=ax, include_values=False, xticks_rotation="vertical",
             cmap="Blues", colorbar=False)
ax.set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

## 10. Predict one image



In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

IMAGE_TO_PREDICT = Path("CV9/bird_dataset/test/001.Black_footed_Albatross/2.jpg")

def predict_image(image_path, trained_model=best_model):
    if image_path is None:
        raise ValueError("Set IMAGE_TO_PREDICT to an image path first.")

    img = load_img(
        image_path,
        target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
        color_mode="rgb",
    )
    x = img_to_array(img).astype("float32") / 255.0
    x = np.expand_dims(x, axis=0)

    probabilities = trained_model.predict(x, verbose=0)[0]
    index = int(np.argmax(probabilities))
    return class_names[index], float(probabilities[index]), probabilities


predicted_class, confidence, all_probabilities = predict_image(IMAGE_TO_PREDICT)
print(predicted_class, f"{confidence:.2%}")